# Establishing correctness by comparing outputs

This notebook aims to present the correctness of the different implementations in a black-box settings to users who want to use the package but don't want to delve into the implementation details. For this purpose, this notebook compares the outputs of different algorithms to a simple exhaustive for loop search. Note that the notbook can take more than 20 minutes of run time.

In [1]:
n_repeat = 1

## Installation

In [ ]:
try:
    from google.colab import files
    colab = True
except ImportError:
    colab = False

if colab and not os.path.exists("XT-neighbor"):
    !git clone https://github.com/heartnetkung/XT-neighbor.git
    repo_path = "XT-neighbor/"
else:
    repo_path = "../"

compile XTNeighbor-streaming

In [ ]:
! mkdir -p {repo_path}/xtneighbor_streaming/build
! cd {repo_path}/xtneighbor_streaming/build; cmake -S .. -B .;make

-- Configuring done (0.0s)
-- Generating done (0.0s)
-- Build files have been written to: /home/andreas/repos/XT-neighbor/2.0/build
[100%] Built target xt_neighbor


compile XTNeighbor

In [ ]:
! mkdir -p {repo_path}xtneighbor/build
! cd {repo_path}xtneighbor/build; cmake -S .. -B .;make

-- Configuring done (0.0s)
-- Generating done (0.0s)
-- Build files have been written to: /home/andreas/repos/XT-neighbor/1.0/build
[100%] Built target xt_neighbor


## Setup and data import

In [4]:
from os import path
import numpy as np
import pandas as pd
import seaborn as sns
import time
import subprocess
import re
import random
from pyrepseq import levenshtein_distance
import pyrepseq
import symscan

read in data

In [5]:
N_FILES=6

data = []
for i in range(1,N_FILES+1):
  data += pd.read_csv(f'data/emerson{i}.zip', compression='zip', header=0)['cdr3'].to_list()

print('first row:', data[0]);
print(f'len: {len(data):,}')

first row: CASSLDSYEQYF
len: 57,472,488


Write files functions used for XTNeighbor and XTNeighbor-streaming respectively

In [6]:
def writeFile(seqs):
  with open("input.txt","w") as file1:
    file1.writelines(seq+'\n' for seq in seqs)

def writeFile2(seqs):
  with open("input2.txt","w") as file1:
    file1.writelines(seq+'\n' for seq in (['cdr3']+seqs))

## Implementations

In [7]:
def xt_neighbor(seqs,threshold,_len): #verbose is ignored
  ! ./1.0/build/xt_neighbor -p "input.txt" -n "$_len" -d "$threshold" -o "xt_output.txt"
  return read_result('xt_output.txt')

def xt_neighbor_streaming(seqs,threshold,_len):
  ! ./2.0/build/xt_neighbor -i "input2.txt" -n "$_len" -d "$threshold" -o "xt_streaming_output.txt"
  return read_result('xt_streaming_output.txt')

def symdel(seqs,threshold,_len):
  return set(pyrepseq.symdel(seqs,max_edits=threshold, output_symmetric=False))

def run_symscan(seqs,threshold,_len):
  row, col, dists = symscan.get_neighbors_within(seqs, max_distance=threshold)
  return set(zip(row, col, dists))

def for_loop(seqs,threshold,_len):
  ans = set()
  for i in range(len(seqs)):
    for j in range(len(seqs)):
        if i >= j:
            continue
        dist = levenshtein_distance(seqs[j], seqs[i], score_cutoff=threshold)
        if dist <= threshold:
            ans.add((i, j, dist))
  return ans

def prepare(seqs):
  writeFile(seqs)
  writeFile2(seqs)

def read_result(filename):
  df = pd.read_csv(filename, sep=' ', header=None)
  row_set = set(df.itertuples(index=False, name=None))
  return row_set


## Run on small datasets

In [8]:
algorithms = {
    'for_loop':for_loop,
    'symdel':symdel,
    'symscan':run_symscan,
    'xt':xt_neighbor,
    'xt_streaming':xt_neighbor_streaming,

}

def perform(subset, distance, algorithms, size):
    result=None
    for alg_name in algorithms:
        print('running algorithm:', alg_name)
        prepare(subset)
        new_result = algorithms[alg_name](subset,distance,size)
        if result is None:
            result = new_result
        elif result != new_result:
            raise Exception('comparison failed')


def run_exp(distance, size, shuffle=True):
    for i in range(n_repeat):
        subset = random.Random(i).sample(data,size)
        perform(subset, distance, algorithms, size)
    print('success!')

In [9]:
run_exp(distance=1, size=5000)

running algorithm: for_loop
running algorithm: symdel
running algorithm: symscan
running algorithm: xt
running algorithm: xt_streaming
success!


In [10]:
run_exp(distance=2, size=5000)

running algorithm: for_loop


running algorithm: symdel
running algorithm: symscan
running algorithm: xt
running algorithm: xt_streaming
success!


In [11]:
run_exp(distance=3, size=5000)

running algorithm: for_loop


running algorithm: symdel
running algorithm: symscan
running algorithm: xt
running algorithm: xt_streaming
success!


## Run on large datasets

In [12]:
algorithms = {
    'xt_streaming':xt_neighbor_streaming,
    'symscan':run_symscan,
}

In [13]:
run_exp(distance=1, size=3_000_000)

running algorithm: xt_streaming
running algorithm: symscan
success!


In [14]:
run_exp(distance=2, size=1_000_000)

running algorithm: xt_streaming
running algorithm: symscan
success!


In [15]:
run_exp(distance=3, size=100_000)

running algorithm: xt_streaming
running algorithm: symscan
success!
